# Control de Lectura 3 — Verificación numérica de gradientes

**Objetivo.** Contrastar dos gradientes candidatos con diferencias finitas centrales e interpretar el efecto del paso `h` y de la precisión numérica, sin derivar a mano ni entrenar.

- Función: `f(w) = w**3 - 2*w`
- Candidatos a derivada: `gA(w) = 3*w**2 - 2` y `gB(w) = 3*w - 2`
- Punto de verificación: `w = 2.0`, con los pasos `h = 1e-1`, `h = 1e-5` y `h = 1e-16` (los tres casos se conservan en la evidencia).

El notebook registra primero la predicción P0 (antes de ejecutar), completa la función `diferencia_central`, ejecuta la evidencia numérica con floats de Python sin redondear cálculos, responde las preguntas de interpretación P1–P3 y deja preparada la entrega `resultados_control_3.csv`.

## Predicción previa (P0)

*Respuesta registrada antes de ejecutar cualquier celda de cómputo; se conserva tal cual, aunque después deba corregirse.*

**¿Qué dos cantidades se comparan al verificar un gradiente?** Se compara el valor del candidato `g(w)` evaluado en `w` contra la aproximación numérica de la derivada de `f` en `w` construida por diferencias finitas, es decir, la pendiente entre `f(w+h)` y `f(w-h)`.

**¿Un `h` menor siempre mejora la aproximación?** No. Al reducir `h` disminuye el error de truncamiento (del orden de `h**2` en el método central), pero crece el error de redondeo del float por cancelación, porque `w+h` y `w-h` se acercan al espaciado de la máquina hasta dejar de distinguirse de `w`. Debe existir, entonces, un `h` intermedio óptimo: demasiado grande domina el truncamiento; demasiado pequeño domina el redondeo.

Esta predicción se contrasta con la evidencia en la pregunta P2.

## Implementación de diferencia_central

De la función `diferencia_central(f, w, h)` se completa únicamente la expresión pendiente, usando el método central estudiado:

```python
return (f(w + h) - f(w - h)) / (2 * h)
```

Dentro de la función no se usa la derivada analítica ni ninguno de los candidatos `gA` o `gB`; el resto del código (definición de `f`, `gA` y `gB`, los tres casos, la tabla y el CSV) está proporcionado. Todos los cálculos se realizan con floats de Python y sin redondear valores intermedios: el formateo se aplica solo al mostrar.

## Evidencia numérica (w=2.0; h=1e-1, 1e-5, 1e-16)

La siguiente celda ejecuta los tres casos pedidos con `w = 2.0` y los pasos `h = 1e-1`, `1e-5` y `1e-16`, y mantiene los tres en la evidencia, incluido el caso extremo `h = 1e-16`. Para cada fila se observan la aproximación central, los candidatos `gA(2.0)` y `gB(2.0)`, las discrepancias absolutas, los puntos perturbados `w+h` y `w-h` con su representación completa (sin redondeo) y su igualdad real (`w+h == w`, `w-h == w` y `f(w+h) == f(w-h)`). Al final se construye un DataFrame y se guarda `resultados_control_3.csv` con las columnas requeridas.

## Preguntas de interpretación (P1–P3)

**P1. ¿Qué candidato respalda una fila fiable?**

La fila fiable es la del paso intermedio `h = 1e-5`: la aproximación central es `10.000000000198739`, con discrepancia absoluta `|aprox - gA(2.0)| = 1.9873880319210002e-10` y `|aprox - gB(2.0)| = 6.000000000198739`. La aproximación coincide con `gA(2.0) = 10.0` con una discrepancia despreciable, mientras que dista más de seis unidades de `gB(2.0) = 4.0`; por tanto, la evidencia numérica respalda al candidato `gA(w) = 3*w**2 - 2`. La fila `h = 1e-1` también apunta a `gA` (aproximación `10.010000000000007`, discrepancias `0.010000000000006892` y `6.010000000000007`), pero con un error de truncamiento visible; la fila `h = 1e-16` no es fiable, como se explica en P2.

**P2. Comparación entre el menor `h` y el intermedio.**

Con `h = 1e-5` la aproximación es `10.000000000198739` y los puntos perturbados sí se distinguen de `w`: `w+h = 2.00001` y `w-h = 1.99999`, con igualdad real falsa en ambos casos. Con el menor paso, `h = 1e-16`, la aproximación colapsa a `0.0`: `w+h` y `w-h` valen exactamente `2.0`, es decir, `w+h == w` y `w-h == w` resultan verdaderos (y también `f(w+h) == f(w-h)`), el numerador se anula y las discrepancias pasan a `10.0` frente a `gA` y `4.0` frente a `gB`. La causa es la pérdida de significancia (cancelación catastrófica): `h = 1e-16` está por debajo de la precisión del float en torno a `2.0` (el espaciado de la máquina), de modo que al sumar o restar el paso el resultado se redondea de vuelta a `w`, los puntos perturbados pierden toda la información del paso y la diferencia central se degenera. Esto confirma la predicción P0: un `h` menor no siempre mejora la aproximación; por debajo de cierto umbral domina el error de redondeo y la aproximación se degrada hasta colapsar, de ahí la conveniencia de un `h` intermedio.

**P3. ¿Coincidir en `w = 2` demuestra corrección para cualquier entrada?**

No. La coincidencia en un solo punto solo muestra compatibilidad en ese punto: dos funciones distintas pueden coincidir en valores aislados de `w` (incluso un candidato incorrecto podría pasar la prueba en un único `w` bien elegido), de modo que el contraste en `w = 2` es evidencia necesaria pero no suficiente de corrección general. Como comprobación adicional, sin programarla: repetir el mismo contraste (diferencia central frente a ambos candidatos) en otros puntos, por ejemplo `w = 0`, `w = -1` y `w = 3`, y para cada punto barrer varios pasos `h`, verificando que la discrepancia con `gA` se mantiene pequeña y la de `gB` grande de forma consistente en todo el barrido.

## Entrega: resultados_control_3.csv y estado

**Estado: terminado, sin partes pendientes.** El notebook incluye el título y contexto, la predicción P0 registrada antes de ejecutar, la función `diferencia_central` completada con el método central, la evidencia de los tres casos (`w = 2.0`; `h = 1e-1`, `1e-5`, `1e-16`) y las respuestas P0–P3.

El archivo `resultados_control_3.csv` es generado por la celda de código de la sección de evidencia, con las columnas `h`, `aproximacion`, `gA`, `gB`, `disc_gA`, `disc_gB`, `w_mas_h`, `w_menos_h`, `igualdad_w_mas_h` e `igualdad_w_menos_h`, y acompaña esta entrega junto con el notebook ejecutado.

### tareas: Script Python con floats nativos de Python, sin redondear cálculos intermedios. Defina f(w) = w**3 - 2*w y los dos candidatos a derivada gA(w) = 3*w**2 - 2 y gB(w) = 3*w - 2. Complete únicamente la expresión pendiente de diferencia_central(f, w, h), que debe devolver (f(w+h) - f(w-h)) / (2*h) (método de diferencia central); no use la derivada analítica ni los candidatos dentro de la función. Con w = 2.0 y los pasos h = 1e-1, 1e-5 y 1e-16 (mantener los tres casos en la evidencia), calcule para cada caso: la aproximación central, gA(2.0), gB(2.0), las discrepancias absolutas |aprox - gA(2.0)| y |aprox - gB(2.0)|, los puntos perturbados w+h y w-h tal como los representa el float (repr completo, sin redondeo) y su igualdad real (w+h == w, w-h == w y f(w+h) == f(w-h)). Imprima una tabla legible con las tres filas (formatee solo al mostrar). Construya un DataFrame y guárdelo como resultados_control_3.csv con columnas: h, aproximacion, gA, gB, disc_gA, disc_gB, w_mas_h, w_menos_h, igualdad_w_mas_h, igualdad_w_menos_h.

In [1]:
# -*- coding: utf-8 -*-
"""
CONTROL DE LECTURA 3 - Verificacion numerica de gradientes (subtarea s1).

f(w)  = w**3 - 2*w
gA(w) = 3*w**2 - 2   (candidato A)
gB(w) = 3*w - 2      (candidato B)

Se completa unicamente la expresion pendiente de diferencia_central(f, w, h)
con el metodo de diferencia central: (f(w+h) - f(w-h)) / (2*h).
Dentro de esa funcion NO se usa la derivada analitica ni los candidatos gA/gB.

Casos: w = 2.0 con h = 1e-1, 1e-5, 1e-16, con float nativo de Python y sin
redondear calculos intermedios (el formateo se aplica solo al mostrar/guardar).

Salidas:
  * tabla legible en stdout (tres filas, una por cada h)
  * resultados_control_3.csv (una fila por cada h)
  * resultados.json (todos los numeros calculados, como floats nativos)
"""

import csv
import json

# ---------------------------------------------------------------------------
# Tarea 1: prediccion P0 registrada ANTES de ejecutar (se conserva en evidencia)
# ---------------------------------------------------------------------------
PREDICCION_P0 = (
    "P0 (antes de ejecutar): al verificar un gradiente se comparan la "
    "aproximacion numerica por diferencia central con el valor de cada "
    "candidato en w. Un h menor no siempre mejora la aproximacion: reduce el "
    "error de truncamiento (~h^2), pero por debajo de cierto tamano domina el "
    "error de redondeo del float y la aproximacion se degrada o colapsa "
    "(p. ej., cuando w+h o w-h dejan de distinguirse de w)."
)

# ---------------------------------------------------------------------------
# Funciones proporcionadas por el enunciado
# ---------------------------------------------------------------------------
def f(w):
    return w**3 - 2*w


def gA(w):
    return 3*w**2 - 2


def gB(w):
    return 3*w - 2


# ---------------------------------------------------------------------------
# Tarea 2: funcion a completar. UNICAMENTE la expresion pendiente:
# metodo de diferencia central. Sin derivada analitica y sin gA/gB.
# ---------------------------------------------------------------------------
def diferencia_central(f, w, h):
    # Expresion pendiente (metodo de diferencia central):
    return (f(w + h) - f(w - h)) / (2 * h)


# ---------------------------------------------------------------------------
# Tarea 3: ejecucion de los tres casos (float de Python, sin redondear)
# ---------------------------------------------------------------------------
w = 2.0
pasos = [1e-1, 1e-5, 1e-16]

f_w = f(w)     # 4.0
gA_w = gA(w)   # 10.0
gB_w = gB(w)   # 4.0

casos = []
for h in pasos:
    aproximacion = diferencia_central(f, w, h)  # (f(w+h) - f(w-h)) / (2*h)
    disc_gA = abs(aproximacion - gA_w)
    disc_gB = abs(aproximacion - gB_w)
    w_mas_h = w + h        # punto perturbado tal como lo representa el float
    w_menos_h = w - h
    casos.append({
        "h": h,
        "aproximacion": aproximacion,
        "gA": gA_w,
        "gB": gB_w,
        "disc_gA": disc_gA,
        "disc_gB": disc_gB,
        "w_mas_h": w_mas_h,
        "w_menos_h": w_menos_h,
        "igualdad_w_mas_h": bool(w_mas_h == w),
        "igualdad_w_menos_h": bool(w_menos_h == w),
        "igualdad_f_perturbados": bool(f(w_mas_h) == f(w_menos_h)),
    })

# ---------------------------------------------------------------------------
# Tabla legible en stdout (el formateo/repr se aplica solo al mostrar)
# ---------------------------------------------------------------------------
encabezados = ["h", "aproximacion", "gA", "gB", "disc_gA", "disc_gB",
               "w+h (repr)", "w-h (repr)", "w+h==w", "w-h==w",
               "f(w+h)==f(w-h)"]

filas_tabla = []
for c in casos:
    filas_tabla.append([
        repr(c["h"]),
        repr(c["aproximacion"]),
        repr(c["gA"]),
        repr(c["gB"]),
        repr(c["disc_gA"]),
        repr(c["disc_gB"]),
        repr(c["w_mas_h"]),
        repr(c["w_menos_h"]),
        str(c["igualdad_w_mas_h"]),
        str(c["igualdad_w_menos_h"]),
        str(c["igualdad_f_perturbados"]),
    ])

anchos = [max([len(encabezados[i])] + [len(r[i]) for r in filas_tabla])
          for i in range(len(encabezados))]
separador = "-+-".join("-" * a for a in anchos)

print("CONTROL DE LECTURA 3 - Verificacion de gradientes por diferencia central")
print("f(w) = w**3 - 2*w   |   gA(w) = 3*w**2 - 2   |   gB(w) = 3*w - 2")
print("w = %r   ->   f(2.0) = %r, gA(2.0) = %r, gB(2.0) = %r"
      % (w, f_w, gA_w, gB_w))
print()
print(separador)
print(" | ".join(encabezados[i].ljust(anchos[i]) for i in range(len(encabezados))))
print(separador)
for r in filas_tabla:
    print(" | ".join(r[i].ljust(anchos[i]) for i in range(len(encabezados))))
print(separador)

# ---------------------------------------------------------------------------
# DataFrame -> resultados_control_3.csv (una fila por cada h)
# pandas puede no estar disponible en el sandbox; se usa si existe y, si no,
# se escribe un CSV equivalente con el modulo estandar csv.
# ---------------------------------------------------------------------------
columnas_csv = ["h", "aproximacion", "gA", "gB", "disc_gA", "disc_gB",
                "w_mas_h", "w_menos_h", "igualdad_w_mas_h",
                "igualdad_w_menos_h"]

csv_escrito = False
try:
    import pandas as pd
    df = pd.DataFrame([{k: c[k] for k in columnas_csv} for c in casos],
                      columns=columnas_csv)
    df.to_csv("resultados_control_3.csv", index=False)
    csv_escrito = True
except Exception:
    csv_escrito = False

if not csv_escrito:
    with open("resultados_control_3.csv", "w", newline="",
              encoding="utf-8") as fh:
        escritor = csv.writer(fh)
        escritor.writerow(columnas_csv)
        for c in casos:
            escritor.writerow([
                repr(c["h"]),
                repr(c["aproximacion"]),
                repr(c["gA"]),
                repr(c["gB"]),
                repr(c["disc_gA"]),
                repr(c["disc_gB"]),
                repr(c["w_mas_h"]),
                repr(c["w_menos_h"]),
                str(c["igualdad_w_mas_h"]),
                str(c["igualdad_w_menos_h"]),
            ])

# ---------------------------------------------------------------------------
# resultados.json: todos los numeros calculados, como floats nativos de Python
# ---------------------------------------------------------------------------
resultados = {
    "subtarea": "s1_control_lectura_3_gradientes",
    "prediccion_P0": PREDICCION_P0,
    "w": w,
    "f_en_w": f_w,
    "gA_en_w": gA_w,
    "gB_en_w": gB_w,
    "casos": [
        {
            "h": c["h"],
            "aproximacion": c["aproximacion"],
            "gA": c["gA"],
            "gB": c["gB"],
            "disc_gA": c["disc_gA"],
            "disc_gB": c["disc_gB"],
            "w_mas_h": c["w_mas_h"],
            "w_menos_h": c["w_menos_h"],
            "repr_w_mas_h": repr(c["w_mas_h"]),
            "repr_w_menos_h": repr(c["w_menos_h"]),
            "igualdad_w_mas_h": c["igualdad_w_mas_h"],
            "igualdad_w_menos_h": c["igualdad_w_menos_h"],
            "igualdad_f_perturbados": c["igualdad_f_perturbados"],
        }
        for c in casos
    ],
}

with open("resultados.json", "w", encoding="utf-8") as fh:
    json.dump(resultados, fh, indent=2)

print()
print("Contenido de resultados.json:")
print(json.dumps(resultados, indent=2))
print()
print("Archivos generados: resultados_control_3.csv, resultados.json")


CONTROL DE LECTURA 3 - Verificacion de gradientes por diferencia central
f(w) = w**3 - 2*w   |   gA(w) = 3*w**2 - 2   |   gB(w) = 3*w - 2
w = 2.0   ->   f(2.0) = 4.0, gA(2.0) = 10.0, gB(2.0) = 4.0

------+--------------------+------+-----+------------------------+-------------------+------------+------------+--------+--------+---------------
h     | aproximacion       | gA   | gB  | disc_gA                | disc_gB           | w+h (repr) | w-h (repr) | w+h==w | w-h==w | f(w+h)==f(w-h)
------+--------------------+------+-----+------------------------+-------------------+------------+------------+--------+--------+---------------
0.1   | 10.010000000000007 | 10.0 | 4.0 | 0.010000000000006892   | 6.010000000000007 | 2.1        | 1.9        | False  | False  | False         
1e-05 | 10.000000000198739 | 10.0 | 4.0 | 1.9873880319210002e-10 | 6.000000000198739 | 2.00001    | 1.99999    | False  | False  | False         
1e-16 | 0.0                | 10.0 | 4.0 | 10.0                   | 4.0  